In [1]:
import sys
from pathlib import Path

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(str(Path.cwd().parent))

In [2]:
ideal_readme = """
Project overview and description.
Business problem or motivation.
Dataset or input data description.
Installation instructions.
Setup guide.
Usage examples.
How to run the project.
Project structure.
Technologies and tech stack.
Modeling approach or implementation details.
Evaluation metrics and results.
Screenshots, plots, demo images or examples.
Recommendations and future improvements.
"""

In [3]:
readme_examples = [
    {
        "repo": "strong_ml_project",
        "text": """
        # Credit Risk Scoring

        This project builds a machine learning model for credit risk scoring.

        ## Project Overview
        The goal is to predict default probability using tabular financial data.

        ## Dataset
        The dataset contains client and loan application features.

        ## Installation
        pip install -r requirements.txt

        ## Usage
        python -m src.train
        streamlit run app/streamlit_app.py

        ## Technologies
        Python, pandas, scikit-learn, XGBoost, SHAP, Streamlit

        ## Results
        Logistic Regression achieved ROC-AUC of 0.804.
        The project includes confusion matrix, ROC curves and SHAP explanations.

        ## Project Structure
        src, notebooks, app, tests, reports
        """,
    },
    {
        "repo": "medium_project",
        "text": """
        # Data Analysis Project

        This repository contains a data analysis project.

        It uses Python and pandas.

        To run:
        python main.py
        """,
    },
    {
        "repo": "weak_project",
        "text": """
        # My project

        Some code.
        """,
    },
    {
        "repo": "web_project",
        "text": """
        # FastAPI Backend

        Backend service built with FastAPI and PostgreSQL.

        ## Installation
        pip install -r requirements.txt

        ## Usage
        uvicorn app.main:app --reload

        ## Technologies
        Python, FastAPI, PostgreSQL, Docker
        """,
    },
]

In [4]:
readme_df = pd.DataFrame(readme_examples)

readme_df

,repo,text
0,strong_ml_project,\n # Credit Risk Scoring\n\n Thi...
1,medium_project,\n # Data Analysis Project\n\n T...
2,weak_project,\n # My project\n\n Some code.\n...
3,web_project,\n # FastAPI Backend\n\n Backend...


In [5]:
documents = [ideal_readme] + readme_df["text"].tolist()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
)

tfidf_matrix = vectorizer.fit_transform(documents)

tfidf_matrix.shape

(5, 226)

In [6]:
similarities = cosine_similarity(
    tfidf_matrix[0],
    tfidf_matrix[1:],
).flatten()

readme_df["similarity_to_ideal"] = similarities
readme_df["nlp_readme_score"] = (readme_df["similarity_to_ideal"] * 100).round().astype(int)

readme_df[["repo", "similarity_to_ideal", "nlp_readme_score"]].sort_values(
    "nlp_readme_score",
    ascending=False,
)

,repo,similarity_to_ideal,nlp_readme_score
0,strong_ml_project,0.098825,10
1,medium_project,0.067247,7
2,weak_project,0.072711,7
3,web_project,0.021783,2


In [7]:
feature_names = vectorizer.get_feature_names_out()

ideal_vector = tfidf_matrix[0].toarray().flatten()

top_terms = pd.DataFrame(
    {
        "term": feature_names,
        "tfidf": ideal_vector,
    }
).sort_values("tfidf", ascending=False)

top_terms.head(20)

,term,tfidf
50,description,0.232488
58,examples,0.232488
136,project,0.196469
25,business problem,0.116244
60,examples run,0.116244
24,business,0.116244
77,images examples,0.116244
76,images,0.116244
79,implementation details,0.116244
59,examples recommendations,0.116244


In [8]:
def calculate_nlp_readme_score(readme_text: str, reference_text: str = ideal_readme) -> int:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
    )

    matrix = vectorizer.fit_transform([reference_text, readme_text])
    similarity = cosine_similarity(matrix[0], matrix[1])[0][0]

    return round(similarity * 100)

In [9]:
for example in readme_examples:
    score = calculate_nlp_readme_score(example["text"])
    print(example["repo"], score)

strong_ml_project 11
medium_project 8
weak_project 11
web_project 2


## Выводы

В этом ноутбуке был протестирован простой NLP-подход для оценки качества README.

Идея основана на сравнении README проекта с эталонным текстом хорошего portfolio README.  
Для этого использовались TF-IDF-векторизация и cosine similarity.

Полученный similarity score можно использовать как дополнительный README NLP score.  
Он не заменяет rule-based проверки, но дополняет их: помогает оценить, насколько README похож на хорошо оформленное описание проекта.

Дальше эту логику можно вынести в `src/readme_nlp.py` и интегрировать в общий scoring pipeline.